### Thermo Sudoku LP Solver
Base code is adapted from https://github.com/Lakshmi-1212/Sudoku_Solver_LP.git

In [1]:
import pulp as plp

In [2]:
def add_default_sudoku_constraints(prob, grid_vars, rows, cols, grids, values):
    
    # Constraint to ensure only one value is filled for a cell
    for row in rows:
        for col in cols:
                prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[row][col][value] for value in values]),
                                        sense=plp.LpConstraintEQ, rhs=1, name=f"constraint_sum_{row}_{col}"))


    # Constraint to ensure that values from 1 to 9 is filled only once in a row        
    for row in rows:
        for value in values:
            prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[row][col][value]*value  for col in cols]),
                                        sense=plp.LpConstraintEQ, rhs=value, name=f"constraint_uniq_row_{row}_{value}"))

    # Constraint to ensure that values from 1 to 9 is filled only once in a column        
    for col in cols:
        for value in values:
            prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[row][col][value]*value  for row in rows]),
                                        sense=plp.LpConstraintEQ, rhs=value, name=f"constraint_uniq_col_{col}_{value}"))


    # Constraint to ensure that values from 1 to 9 is filled only once in the 3x3 grid       
    for grid in grids:
        grid_row  = int(grid/3)
        grid_col  = int(grid%3)

        for value in values:
            prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[grid_row*3+row][grid_col*3+col][value]*value  for col in range(0,3) for row in range(0,3)]),
                                        sense=plp.LpConstraintEQ, rhs=value, name=f"constraint_uniq_grid_{grid}_{value}"))

In [3]:
def add_diagonal_sudoku_constraints(prob, grid_vars, rows, cols, values):
    
        # Constraint from top-left to bottom-right - numbers 1 - 9 should not repeat
        for value in values:
                prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[row][row][value]*value  for row in rows]),
                                            sense=plp.LpConstraintEQ, rhs=value, name=f"constraint_uniq_diag1_{value}"))


        # Constraint from top-right to bottom-left - numbers 1 - 9 should not repeat
        for value in values:
                prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[row][len(rows)-row-1][value]*value  for row in rows]),
                                            sense=plp.LpConstraintEQ, rhs=value, name=f"constraint_uniq_diag2_{value}"))


In [4]:
def add_thermo_sudoku_constraints(prob, grid_vars, rows, cols, values, thermo_info):
    thermometer_idx = 0
    for thermometer in thermo_info:
        for i in range(len(thermometer) - 1):
            prev_row, prev_col = thermometer[i][0] - 1, thermometer[i][1] - 1
            next_row, next_col = thermometer[i + 1][0] - 1, thermometer[i + 1][1] - 1

            prev_val = plp.lpSum([grid_vars[prev_row][prev_col][value] * value for value in values])
            next_val = plp.lpSum([grid_vars[next_row][next_col][value] * value for value in values])

            prob.addConstraint(plp.LpConstraint(e=next_val - prev_val, 
                                                sense=plp.LpConstraintGE,
                                                rhs=1,
                                                name=f"Thermo_{thermometer_idx}_{i}"))
        thermometer_idx += 1

In [5]:
def add_prefilled_constraints(prob, input_sudoku, grid_vars, rows, cols, values):
    for row in rows:
        for col in cols:
            if(input_sudoku[row][col] != 0):
                prob.addConstraint(plp.LpConstraint(e=plp.lpSum([grid_vars[row][col][value]*value  for value in values]), 
                                                    sense=plp.LpConstraintEQ, 
                                                    rhs=input_sudoku[row][col],
                                                    name=f"constraint_prefilled_{row}_{col}"))

In [6]:
# Extract solution from the target variable to a list array

def extract_solution(grid_vars, rows, cols, values):
    solution = [[0 for col in cols] for row in rows]
    grid_list = []
    for row in rows:
        for col in cols:
            for value in values:
                if plp.value(grid_vars[row][col][value]):
                    solution[row][col] = value 
    return solution

In [7]:
# print solution with a grid

def print_solution(solution, rows,cols):
    # Print the final result
    print(f"\nFinal result:")

    print("\n\n+ ----------- + ----------- + ----------- +",end="")
    for row in rows:
        print("\n",end="\n|  ")
        for col in cols:
            num_end = "  |  " if ((col+1)%3 == 0) else "   "
            print(solution[row][col],end=num_end)

        if ((row+1)%3 == 0):
            print("\n\n+ ----------- + ----------- + ----------- +",end="")

In [8]:
# check if the depth of the given sequence data is three

def is_arr_depth_three_helper(arr, depth):
    if not isinstance(arr, list) and not isinstance(arr, tuple):
        return depth == 3

    return all(is_arr_depth_three_helper(next_arr, depth + 1) for next_arr in arr)

def is_arr_depth_three(arr):
    return is_arr_depth_three_helper(arr, 0)

In [9]:
def solve_sudoku(input_sudoku, type = "regular", thermo_info_arr = None):
    if type == "thermo" and thermo_info_arr == None: 
        print("ERROR::solve_sudoku::Thermo info is not provided.")
        return
    
    if type == "thermo" and not is_arr_depth_three(thermo_info_arr):
        print("ERROR::solve_sudoku::Thermo info array must be a 3D array.")
        return

    # Create the linear programming problem
    prob = plp.LpProblem("Sudoku_Solver")

    rows = range(0,9)
    cols = range(0,9)
    grids = range(0,9)
    values = range(1,10)

    # Decision Variable/Target variable
    grid_vars = plp.LpVariable.dicts("grid_value", (rows,cols,values), cat='Binary') 

    # Set the objective function
    # Sudoku works only on the constraints - feasibility problem 
    # There is no objective function that we are trying maximize or minimize.
    # Set a dummy objective
    objective = plp.lpSum(0)
    prob.setObjective(objective)

    # Create the default constraints to solve sudoku
    add_default_sudoku_constraints(prob, grid_vars, rows, cols, grids, values)

    # Add the constraints if type flag is set
    if (type == "diagonal"):
        add_diagonal_sudoku_constraints(prob, grid_vars, rows, cols, values)
    elif (type == "thermo"):
        add_thermo_sudoku_constraints(prob, grid_vars, rows, cols, values, thermo_info_arr)
        
    # Fill the prefilled values from input sudoku as constraints
    add_prefilled_constraints(prob, input_sudoku, grid_vars, rows, cols, values)


    # Solve the problem
    prob.solve(plp.PULP_CBC_CMD(msg=False))

    # Print the status of the solution
    solution_status = plp.LpStatus[prob.status]
    print(f'Solution Status = {plp.LpStatus[prob.status]}')

    # Extract the solution if an optimal solution has been identified
    if solution_status == 'Optimal':
        solution = extract_solution(grid_vars, rows, cols, values)
        print_solution(solution, rows,cols)


----------
## Regular Sudoku Test

In [10]:
regular_sudoku = [
                    [3,0,0, 8,0,0, 0,0,1],
                    [0,0,0, 0,0,2, 0,0,0],
                    [0,4,1, 5,0,0, 8,3,0],

                    [0,2,0, 0,0,1, 0,0,0],
                    [8,5,0, 4,0,3, 0,1,7],
                    [0,0,0, 7,0,0, 0,2,0],
                    
                    [0,8,5, 0,0,9, 7,4,0],
                    [0,0,0, 1,0,0, 0,0,0],
                    [9,0,0, 0,0,7, 0,0,6]
                ]

solve_sudoku(input_sudoku=regular_sudoku)

Solution Status = Optimal

Final result:


+ ----------- + ----------- + ----------- +

|  3   6   7  |  8   9   4  |  2   5   1  |  

|  5   9   8  |  3   1   2  |  6   7   4  |  

|  2   4   1  |  5   7   6  |  8   3   9  |  

+ ----------- + ----------- + ----------- +

|  7   2   3  |  9   8   1  |  4   6   5  |  

|  8   5   6  |  4   2   3  |  9   1   7  |  

|  4   1   9  |  7   6   5  |  3   2   8  |  

+ ----------- + ----------- + ----------- +

|  1   8   5  |  6   3   9  |  7   4   2  |  

|  6   7   2  |  1   4   8  |  5   9   3  |  

|  9   3   4  |  2   5   7  |  1   8   6  |  

+ ----------- + ----------- + ----------- +

----------
## Diagonal Sudoku Test

In [11]:
diagonal_sudoku = [
                    [0,3,0, 2,7,0, 0,0,0],
                    [0,0,0, 0,0,0, 0,0,0],
                    [8,0,0, 0,0,0, 0,0,0],

                    [5,1,0, 0,0,0, 0,8,4],
                    [4,0,0, 5,9,0, 0,7,0],
                    [2,9,0, 0,0,0, 0,1,0],
                    
                    [0,0,0, 0,0,0, 1,0,5],
                    [0,0,6, 3,0,8, 0,0,7],
                    [0,0,0, 0,0,0, 3,0,0]
                ]

solve_sudoku(input_sudoku=diagonal_sudoku, type="diagonal")

Solution Status = Optimal

Final result:


+ ----------- + ----------- + ----------- +

|  6   3   9  |  2   7   4  |  8   5   1  |  

|  1   4   2  |  6   8   5  |  7   3   9  |  

|  8   7   5  |  1   3   9  |  6   4   2  |  

+ ----------- + ----------- + ----------- +

|  5   1   3  |  7   6   2  |  9   8   4  |  

|  4   6   8  |  5   9   1  |  2   7   3  |  

|  2   9   7  |  8   4   3  |  5   1   6  |  

+ ----------- + ----------- + ----------- +

|  3   8   4  |  9   2   7  |  1   6   5  |  

|  9   5   6  |  3   1   8  |  4   2   7  |  

|  7   2   1  |  4   5   6  |  3   9   8  |  

+ ----------- + ----------- + ----------- +

----------
## Thermo Example Test 1

<table style="width:60%">
  <tr>
    <td>Example 1</td>
    <td>Example 1 Solution</td>
  </tr>
  <tr>
    <th><img src="./thermo_sudoku_examples/250905-ThermoSudoku-2.jpg" alt="example1"></th>
    <th><img src="./thermo_sudoku_examples/250905-ThermoSudoku-soln.jpg" alt="example1_soln"></th>
  </tr>
</table>

In [12]:
thermo_sudoku_1 = [
                    [0,0,0, 0,0,3, 0,0,0],
                    [0,0,0, 4,0,6, 0,0,0],
                    [0,0,0, 7,0,0, 0,0,0],

                    [0,0,0, 5,0,0, 0,0,0],
                    [0,0,0, 8,0,1, 0,0,0],
                    [0,0,0, 0,0,4, 0,0,0],
                    
                    [0,0,0, 0,0,5, 0,0,0],
                    [0,0,0, 9,0,2, 0,0,0],
                    [0,0,0, 6,0,0, 0,0,0],
                ]

thermo_info_1 = [
                [(1,6),(1,7),(1,8),(1,9)],    # row 1
                [(2,1),(2,2),(2,3),(2,4)],    # row 2
                [(2,6),(2,7),(2,8),(2,9)],    # row 2
                [(3,1),(3,2),(3,3),(3,4)],    # row 3
                [(4,1),(4,2),(4,3),(4,4)],    # row 4
                [(5,1),(5,2),(5,3),(5,4)],    # row 5
                [(5,6),(5,7),(5,8),(5,9)],    # row 5
                [(6,6),(6,7),(6,8),(6,9)],    # row 6
                [(7,6),(7,7),(7,8),(7,9)],    # row 7
                [(8,1),(8,2),(8,3),(8,4)],    # row 8
                [(8,6),(8,7),(8,8),(8,9)],    # row 8
                [(9,1),(9,2),(9,3),(9,4)],    # row 9
            ]

solve_sudoku(input_sudoku=thermo_sudoku_1, type="thermo", thermo_info_arr=thermo_info_1)

Solution Status = Optimal

Final result:


+ ----------- + ----------- + ----------- +

|  7   8   9  |  1   2   3  |  4   5   6  |  

|  1   2   3  |  4   5   6  |  7   8   9  |  

|  4   5   6  |  7   8   9  |  1   2   3  |  

+ ----------- + ----------- + ----------- +

|  2   3   4  |  5   6   7  |  8   9   1  |  

|  5   6   7  |  8   9   1  |  2   3   4  |  

|  8   9   1  |  2   3   4  |  5   6   7  |  

+ ----------- + ----------- + ----------- +

|  9   1   2  |  3   4   5  |  6   7   8  |  

|  6   7   8  |  9   1   2  |  3   4   5  |  

|  3   4   5  |  6   7   8  |  9   1   2  |  

+ ----------- + ----------- + ----------- +

----------
## Thermo Example Test 2

<table style="width:60%">
  <tr>
    <td>Example 2</td>
    <td>Example 2 Solution</td>
  </tr>
  <tr>
    <th><img src="./thermo_sudoku_examples/251124-ThermoSudoku-CompletelyUnexpectedThermo.jpg" alt="example2"></th>
    <th><img src="./thermo_sudoku_examples/251124-ThermoSudoku-CompletelyUnexpectedThermo-soln.jpg" alt="example2_soln"></th>
  </tr>
</table>

In [13]:
thermo_sudoku_2 = [
                    [0,0,0, 0,0,0, 0,0,0],
                    [0,0,0, 0,0,0, 1,0,0],
                    [0,0,2, 0,0,0, 5,0,0],

                    [0,0,0, 0,3,0, 0,0,1],
                    [0,0,0, 4,0,6, 0,0,0],
                    [9,0,0, 0,7,0, 0,0,0],

                    [0,0,5, 0,0,0, 8,0,0],
                    [0,0,9, 0,0,0, 0,0,0],
                    [0,0,0, 0,0,0, 0,0,0],
                ]

thermo_info_2 = [
                [(2,2),(1,3),(2,4),(1,5),(2,6),(1,7)],    # row 1-2
                [(4,3),(3,4),(2,5)],                      # row 2-4
                [(5,7),(4,7),(3,7)],                      # row 3-5
                [(5,3),(6,3),(7,3)],                      # row 5-7
                [(6,7),(7,6),(8,5)],                      # row 6-8
                [(2,8),(3,9),(4,8),(5,9),(6,8),(7,9)],    # row 2-7
                [(8,2),(7,1),(6,2),(5,1),(4,2),(3,1)],    # row 3-8
                [(8,8),(9,7),(8,6),(9,5),(8,4),(9,3)],    # row 8-9
                ]

solve_sudoku(input_sudoku=thermo_sudoku_2, type="thermo", thermo_info_arr=thermo_info_2)

Solution Status = Optimal

Final result:


+ ----------- + ----------- + ----------- +

|  1   5   4  |  3   6   2  |  9   7   8  |  

|  6   3   7  |  5   9   8  |  1   2   4  |  

|  8   9   2  |  7   4   1  |  5   6   3  |  

+ ----------- + ----------- + ----------- +

|  2   7   6  |  8   3   9  |  4   5   1  |  

|  5   8   1  |  4   2   6  |  3   9   7  |  

|  9   4   3  |  1   7   5  |  6   8   2  |  

+ ----------- + ----------- + ----------- +

|  3   6   5  |  2   1   7  |  8   4   9  |  

|  4   2   9  |  6   8   3  |  7   1   5  |  

|  7   1   8  |  9   5   4  |  2   3   6  |  

+ ----------- + ----------- + ----------- +